# kaggloop Colab worker (GPU)

This notebook is the **GPU half** of the kaggloop compute bridge. Your local Claude Code
enqueues training jobs via `kloop.colab`; they travel through a shared **Google Drive**
folder (`MyDrive/kaggloop/{queue,results}`); this worker runs each job on the GPU and writes
results back for the laptop to ingest.

**Setup (once):**
1. `Runtime -> Change runtime type -> GPU` (T4 is fine; A100/L4 faster).
2. `Runtime -> Run all`.
3. When prompted in cell 2, **upload your `kaggle.json`** (Kaggle -> Account -> Create New API Token).
   It stays in this Colab session only — never written to Drive.
4. Leave this tab open. The last cell polls the queue and runs jobs until you stop it.

Sign in to Colab with the **same Google account whose Drive is synced on the laptop**
(`kazutoki.ando@gmail.com`), so both sides see the same `MyDrive/kaggloop` folder.

In [ ]:
# 1. Mount Google Drive (the shared transport between laptop and Colab).
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT = '/content/drive/MyDrive/kaggloop'   # must match KLOOP_COLAB_* on the laptop
QUEUE = f'{ROOT}/queue'
RESULTS = f'{ROOT}/results'
os.makedirs(QUEUE, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)
print('queue   :', QUEUE)
print('results :', RESULTS)

In [ ]:
# 2. Kaggle credentials. Upload kaggle.json (kept in this session only, not on Drive).
import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
if not os.path.exists('/root/.kaggle/kaggle.json'):
    from google.colab import files
    print('Upload your kaggle.json (Kaggle -> Account -> Create New API Token):')
    up = files.upload()
    for name in up:
        shutil.move(name, '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle credentials configured.')

In [ ]:
# 3. Install the heavy training deps once (torch/numpy are preinstalled on Colab).
#    Each job may still add exact pins via its own requirements.txt.
!pip -q install 'zarr>=3.0.10,<4' geff blosc2 'numcodecs>=0.13,<0.16' \
    scikit-image tifffile imagecodecs tqdm scipy kaggle 2>/dev/null
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

In [ ]:
# 4. Get the worker code (public repo; pulls latest if already cloned).
import os
if not os.path.isdir('/content/kaggloop'):
    !git clone --depth 1 https://github.com/qurore/kaggloop.git /content/kaggloop
else:
    !cd /content/kaggloop && git pull --ff-only
print('worker:', '/content/kaggloop/colab/worker.py')

In [ ]:
# 5. Run the worker. It polls the queue and runs jobs on the GPU until you stop it.
#    First job also downloads + caches the competition data to /content/kaggle_data.
!python /content/kaggloop/colab/worker.py \
    --queue "{QUEUE}" --results "{RESULTS}" \
    --data /content/kaggle_data --loop --interval 20